# Grover search for a solved 4-cube tower (cube 0 fixed, Aer or SV1)

Cube 0 is fixed to one orientation, which leaves 15 qubits for cubes 1 to 3. The same circuit runs on the local Aer simulator or on qBraid's SV1 simulator. Change one line, `DEVICE`, to switch.

## 1. Setup

Set `DEVICE` to `'aer'` for local runs or `'sv1'` for qBraid. SV1 jobs cost credits, so keep `RUNS` small until you've checked the output. Aer is seeded, so its results repeat. SV1 is not seeded, so its counts will vary between runs.

In [ ]:
import math
import numpy as np
from scipy.stats import binomtest
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator

DEVICE = 'aer'         # 'aer' (local) or 'sv1' (qBraid, costs credits)

CUBES = [
    [2, 2, 3, 1, 2, 0],
    [3, 2, 1, 1, 1, 3],
    [0, 2, 2, 1, 3, 3],
    [0, 1, 0, 3, 1, 1],
]

SHOTS = 20000
RUNS = 10
SEED = 2026
BASIS = ['rz', 'sx', 'x', 'cx']
COUNT_GATES = True

## 2. Cube orientations

Two moves, `SPIN` and `TIP`, generate the 24 orientations of a cube. The cell builds the list and checks its length.

In [ ]:
SIDES = [2, 3, 4, 5]        # front, back, left, right
SPIN = [0, 1, 4, 5, 3, 2]
TIP = [2, 3, 1, 0, 4, 5]


def _apply(state, src):
    return tuple(state[s] for s in src)


def _all_orientations():
    start = tuple(range(6))
    seen, frontier = {start}, [start]
    while frontier:
        nxt = []
        for s in frontier:
            for move in (SPIN, TIP):
                t = _apply(s, move)
                if t not in seen:
                    seen.add(t)
                    nxt.append(t)
        frontier = nxt
    return sorted(seen)


ORIENTS = _all_orientations()
assert len(ORIENTS) == 24

## 3. Encoding and the solution check

Each cube's orientation takes 5 bits. Orientations 24 to 31 are invalid. A stack is a solution when each side position shows all four colours across the stack.

In [ ]:
def side_colours(c, o):
    return tuple(CUBES[c][ORIENTS[o][p]] for p in SIDES)


DISPLAY = [[side_colours(c, o) for o in range(24)] for c in range(4)]


def is_solution(idxs):
    if any(o >= 24 for o in idxs):
        return False
    for col in range(4):
        colours = {DISPLAY[c][idxs[c]][col] for c in range(4)}
        if len(colours) != 4:
            return False
    return True


BITS = 5
N_FULL = 4 * BITS


def decode(x):
    return [(x >> (BITS * c)) & 31 for c in range(4)]

## 4. Brute-force the full solution set

This classical pass checks all 2^20 stacks. We use it to pick the anchor orientation and to confirm the reduced problem. The quantum circuit only uses this list through its oracle, so this is not a quantum speedup.

In [ ]:
all_solutions = [x for x in range(2**N_FULL) if is_solution(decode(x))]
print(f'Full search: {len(all_solutions)} valid stacks out of {2**N_FULL}')
assert all_solutions, 'This puzzle has no solution.'

## 5. Fix cube 0 and reduce to 15 qubits

We take the cube 0 orientation from the first solution and keep only the solutions that use it. Cubes 1 to 3 then need 15 qubits. `full_stack` rebuilds the full stack from a 15-bit value.

In [ ]:
anchor = decode(all_solutions[0])[0]
anchor_solutions = [x for x in all_solutions if decode(x)[0] == anchor]

N = 3 * BITS
solutions = [x >> BITS for x in anchor_solutions]
sol_set = set(solutions)


def full_stack(y):
    return [anchor] + [(y >> (BITS * i)) & 31 for i in range(3)]


assert all(is_solution(full_stack(y)) for y in solutions)
k = len(solutions)
print(f'Anchor orientation for cube 0: {anchor}')
print(f'Qubits: {N}, search space: {2**N}, valid stacks with this anchor: {k}')
assert k > 0

## 6. Grover oracle and diffuser

The oracle flips the phase of each solution, and the diffuser reflects amplitudes about their mean. Both act on the 15 qubits.

In [ ]:
def add_oracle(qc):
    for y in solutions:
        zeros = [q for q in range(N) if not (y >> q) & 1]
        qc.x(zeros)
        qc.h(N - 1)
        qc.mcx(list(range(N - 1)), N - 1)
        qc.h(N - 1)
        qc.x(zeros)


def add_diffuser(qc):
    qc.h(range(N))
    qc.x(range(N))
    qc.h(N - 1)
    qc.mcx(list(range(N - 1)), N - 1)
    qc.h(N - 1)
    qc.x(range(N))
    qc.h(range(N))

## 7. Build the Grover circuit

The optimal number of iterations is about (pi/4) times sqrt(N/k). The circuit is built without measurements, and then `measure_all` adds them. The exact probability is computed locally from the statevector as a reference. It costs nothing and doesn't depend on the device.

In [ ]:
iters = math.floor(math.pi / 4 * math.sqrt(2**N / k))
print(f'Grover iterations: {iters}')

qc = QuantumCircuit(N)
qc.h(range(N))
for _ in range(iters):
    add_oracle(qc)
    add_diffuser(qc)

exact = Statevector.from_instruction(qc).probabilities()
print(f'Exact probability of a valid stack (reference): {exact[solutions].sum():.6f}')

qc_meas = qc.copy()
qc_meas.measure_all()

## 8. Run on the chosen device

`run_counts` sends the circuit to Aer or SV1 and returns the bitstring counts. The SV1 branch follows the qBraid SDK pattern from `resources/02_qbraid_sdk.ipynb`: transpile with the rubric basis, submit the job, and wait for the counts. Each SV1 job prints its job ID, which you need for the README.

The bit-order check at the end prepares a known solution and runs it through the device. It costs one more SV1 job, but it confirms that the device's bit order matches our decoding before you trust the numbers.

In [ ]:
def run_counts(circuit, shots, seed):
    if DEVICE == 'aer':
        backend = AerSimulator()
        tqc = transpile(circuit, backend, seed_transpiler=seed)
        return backend.run(tqc, shots=shots, seed_simulator=seed).result().get_counts()
    if DEVICE == 'sv1':
        from qbraid.runtime import QbraidProvider
        device = QbraidProvider().get_device('aws:aws:sim:sv1')
        tqc = transpile(circuit, basis_gates=BASIS, optimization_level=1)
        job = device.run(tqc, shots=shots)
        print(f'  submitted SV1 job {job.id}')
        return job.result().data.get_counts()
    raise ValueError(f"DEVICE must be 'aer' or 'sv1', got {DEVICE!r}")


# Bit-order check: prepare a known solution, measure, and confirm it decodes back.
prep = QuantumCircuit(N)
for q in range(N):
    if (solutions[0] >> q) & 1:
        prep.x(q)
prep.measure_all()
check = run_counts(prep, shots=100, seed=SEED)
top_check = max(check, key=check.get)
assert int(top_check, 2) == solutions[0], 'Bit order mismatch: do not trust the counts'
print(f'Bit-order check passed on {DEVICE}')

## 9. Success rate and p-value

Each run submits `SHOTS` shots with its own seed. On Aer the seed makes the run repeatable. On SV1 each run is a separate job, so the counts will vary.

- **Success rate:** the fraction of shots that decode to a valid stack.
- **Baseline:** k / 2^N, the chance a uniformly random 15-bit value is a valid stack.
- **p-value:** a one-sided binomial test against that baseline. Very small values underflow to 0 in the printout.
- **Pooled row:** all runs combined, with a 95% Wilson interval.

In [ ]:
def wilson_ci(hits, n, z=1.96):
    p = hits / n
    denom = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denom
    return centre - half, centre + half


baseline = k / 2**N
print(f'Device: {DEVICE}')
print(f'Random-guess baseline p0 = {k}/2^{N} = {baseline:.3e}')
print()

run_hits = []
first_counts = None
for r in range(RUNS):
    counts = run_counts(qc_meas, SHOTS, SEED + r)
    if r == 0:
        first_counts = counts
    hits = sum(c for bits, c in counts.items() if int(bits, 2) in sol_set)
    run_hits.append(hits)
    p_run = binomtest(hits, SHOTS, baseline, alternative='greater').pvalue
    print(f'run {r + 1:2d}: P(correct) = {hits / SHOTS:.4f}   p-value = {p_run:.3g}')

total = sum(run_hits)
n_total = RUNS * SHOTS
p_pool = binomtest(total, n_total, baseline, alternative='greater').pvalue
lo, hi = wilson_ci(total, n_total)
print()
print(f'Pooled: P(correct) = {total / n_total:.4f}  95% CI [{lo:.4f}, {hi:.4f}]  p-value = {p_pool:.3g}')

## 10. Decode and check the most common outcome

We take the most frequent outcome from run 1, rebuild the full stack, and check it with `is_solution`. That check does not use the circuit.

In [ ]:
top = max(first_counts, key=first_counts.get)
y = int(top, 2)
idxs = full_stack(y)
print(f'Most common outcome: {top} (seen {first_counts[top]} of {SHOTS} shots)')
print(f'Valid stack: {is_solution(idxs)}')
print('rows: front, back, left, right; columns: cubes')
print(np.array([DISPLAY[c][idxs[c]] for c in range(4)]).T)

## 11. Qubits, two-qubit gates and depth

The rubric counts two-qubit gates after transpiling to `rz`, `sx`, `x` and `cx`. This step is local and doesn't submit a job. Set `COUNT_GATES = False` in the setup cell to skip it.

In [ ]:
if COUNT_GATES:
    tqc = transpile(qc_meas, basis_gates=BASIS, seed_transpiler=SEED)
    print(f'Qubits: {N}')
    print(f"Two-qubit gates (cx): {tqc.count_ops().get('cx', 0)}")
    print(f'Depth: {tqc.depth()}')
else:
    print('Gate counting skipped (COUNT_GATES = False)')